In [46]:
import numpy as np
import pandas as pd

df = pd.read_excel('model_orders.xlsx')
df.head()
# 1. ОПРЕДЕЛЯЕМ КОЛОНКИ (Добавляем всё, что нужно для отчета)
base_cols = ['Артикул', 'Наименование', 'Артикул поставщика', 'Поставщик', 
             'Свободный Ост', 'В ПУТИ', 'Норматив', 'Кол-во на паллете']

# Находим даты динамически
date_columns = [c for c in df.columns if '2024' in str(c)]
final_cols = base_cols + date_columns

# 2. ФИЛЬТРУЕМ И СОЗДАЕМ ТАБЛИЦУ
df_final = df[df['Поставщик'] == 'ХАЯТ МАРКЕТИНГ ООО'][final_cols].copy()

# Находим конкретные месяцы для расчета
col_apr = [c for c in df_final.columns if '2024-04' in str(c)]
col_mar = [c for c in df_final.columns if '2024-03' in str(c)]
col_feb = [c for c in df_final.columns if '2024-02' in str(c)]

# 3. РАСЧЕТЫ (Simple и Smart) с защитой от дублей колонок

# Берем ПЕРВЫЙ элемент из каждого списка [0]
c_apr = col_apr[0]
c_mar = col_mar[0]
c_feb = col_feb[0]

# Простой расчет (только апрель)
df_final['Daily_Sales_Simple'] = (df_final[c_apr] / 30).round(2)
df_final['Order_Pcs'] = np.ceil(
    (df_final['Daily_Sales_Simple'] * df_final['Норматив'] - (df_final['Свободный Ост'] + df_final['В ПУТИ'])) 
    / df_final['Кол-во на паллете'].replace(0, 1)
).clip(lower=0) * df_final['Кол-во на паллете']

# Умный расчет (взвешенное среднее) - ТЕПЕРЬ БЕЗ ОШИБКИ
df_final['Smart_Daily_Sales'] = ((df_final[c_apr]*0.5 + df_final[c_mar]*0.3 + df_final[c_feb]*0.2) / 30).round(2)

df_final['Smart_Order_Pcs'] = np.ceil(
    (df_final['Smart_Daily_Sales'] * df_final['Норматив'] - (df_final['Свободный Ост'] + df_final['В ПУТИ'])) 
    / df_final['Кол-во на паллете'].replace(0, 1)
).clip(lower=0) * df_final['Кол-во на паллете']


# 4. ВЫВОД СРАВНЕНИЯ (Теперь всё точно найдется!)
comparison = df_final[df_final['Order_Pcs'] != df_final['Smart_Order_Pcs']]

print(f"Найдено {len(comparison)} позиций с разным прогнозом заказа.")
comparison[['Артикул', 'Наименование', 'Артикул поставщика', 'Daily_Sales_Simple', 'Smart_Daily_Sales', 'Order_Pcs', 'Smart_Order_Pcs']]


Найдено 20 позиций с разным прогнозом заказа.


,Артикул,Наименование,Артикул поставщика,Daily_Sales_Simple,Smart_Daily_Sales,Order_Pcs,Smart_Order_Pcs
488,450742,"Бумага туалетная ""Focus"" Extra 2-сл., 48 м, 6 ...",5067596,22.23,17.02,216,0
489,450743,"Бумага туалетная ""Focus"" Optimum 2-сл., 22 м, ...",5036770,211.23,179.62,2016,1512
492,450744,"Бумага туалетная ""Focus"" Economic Choice 2-сл....",5056378,176.53,159.39,1440,1152
493,107909,"Бумага туалетная ""Focus"" Premium 3-сл., бел., ...",5080461,29.77,23.21,576,288
495,450774,"Бумага туалетная для диспенсеров 230х108 мм ""F...",5049979,186.60,110.66,3240,2160
497,108404,Бумага туалетная для диспенсеров с центр. вытя...,5036915,27.13,20.27,600,0
502,451423,"Полотенца бумажные ""Familia"" Радуга 2-сл., бел...",5050455,436.30,383.97,960,0
505,112212,Полотенца бумажные для диспенсеров 230х230 мм ...,5083741,0.00,0.12,0,810
506,109909,Полотенца бумажные для диспенсеров 230х205 мм ...,5049975,606.60,435.84,9000,5400
509,109907,Полотенца бумажные для диспенсеров 230х205 мм ...,5049974,351.33,261.08,3300,1980


In [48]:
# Оставляем только нужные колонки для закупки
export_cols = ['Артикул', 'Артикул поставщика', 'Наименование', 'Кол-во на паллете', 'Smart_Order_Pcs']

# Оставляем только строки, где заказ > 0
final_order = df_final[df_final['Smart_Order_Pcs'] > 0][export_cols]

# Сохраняем в Excel
final_order.to_excel('Final_Order_to_Supplier.xlsx', index=False)
print("✅ Файл 'Final_Order_to_Supplier.xlsx' готов!")


✅ Файл 'Final_Order_to_Supplier.xlsx' готов!
